# StandUp4AI: 1000-Video Evaluation (Fixed Labels)

Uses EMNLP dataset laugh labels from seq-Standup4AI/dataset/ — NOT the labels/ folder.
Cell 1: Mount Drive
Cell 2: Scan audio + EMNLP label files, find overlap
Cell 3: Feature extraction + evaluate

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓")

In [ ]:
# Cell 2: Scan audio + EMNLP labels, find overlap
import os, json, warnings, io
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import librosa
import torch, torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score
warnings.filterwarnings('ignore')

BASE = "/content/drive/MyDrive/standup4ai"
AUDIO_DIRS = [f"{BASE}/audio", f"{BASE}/audio_1000"]
OUT_DIR = f"{BASE}/eval_1000"
MODEL_PATH = f"{BASE}/models/top200_prosody_model.pt"
os.makedirs(OUT_DIR, exist_ok=True)

# --- scan audio files ---
audio_files = {}
for d in AUDIO_DIRS:
    p = Path(d)
    if not p.exists():
        print(f"[WARN] missing: {d}"); continue
    for ext in ('*.m4a','*.mp3','*.wav'):
        for f in p.glob(ext):
            audio_files[f.stem] = str(f)
print(f"Audio files: {len(audio_files)}")

# --- scan EMNLP label files recursively ---
EMNLP_ROOT = f"{BASE}/seq-Standup4AI/dataset"
label_files = {}  # vid -> {'path': str, 'split': str}
for f in Path(EMNLP_ROOT).rglob('*.csv'):
    vid = f.stem
    if vid in label_files: continue
    split = 'unknown'
    for part in f.parts:
        if part in ('train','val','test','all'): split = part; break
    label_files[vid] = {'path': str(f), 'split': split}
print(f"EMNLP label files: {len(label_files)}")

# split breakdown
from collections import Counter
split_counts = Counter(v['split'] for v in label_files.values())
for sp, cnt in sorted(split_counts.items()): print(f"  {sp}: {cnt}")

# --- find overlap: audio AND label ---
labeled_audio = set(audio_files.keys()) & set(label_files.keys())
val_vids   = sorted(v for v in labeled_audio if label_files[v]['split']=='val')
train_vids = sorted(v for v in labeled_audio if label_files[v]['split'] in ('train','all'))
test_vids  = sorted(v for v in labeled_audio if label_files[v]['split']=='test')

print(f"\nOverlap (has audio+label): {len(labeled_audio)}")
print(f"  val:   {len(val_vids)}")
print(f"  train: {len(train_vids)}")
print(f"  test:  {len(test_vids)}")
print(f"  all:   {len([v for v in labeled_audio if label_files[v]['split']=='all'])}")

# save for next cell
pd.DataFrame({
    'vid': sorted(labeled_audio),
    'split': [label_files[v]['split'] for v in sorted(labeled_audio)]
}).to_csv(f"{OUT_DIR}/eval_partition.csv", index=False)
print(f"Saved partition: {OUT_DIR}/eval_partition.csv")

In [ ]:
# Cell 3: Feature extractor (15-dim prosody)
def extract_features(path, sr=22050):
    y, sr = librosa.load(path, sr=sr, mono=True)
    if len(y) < 0.5 * sr: return None
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    spec_bw   = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    spec_roll = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    zcr       = np.mean(librosa.feature.zero_crossing_rate(y))
    flatness  = np.mean(librosa.feature.spectral_flatness(y=y))
    rms       = np.mean(librosa.feature.rms(y=y))
    try:
        f0 = librosa.pyin(y, fmin=50, fmax=300, sr=sr)[0]
        f0_c = f0[~np.isnan(f0)]
        if len(f0_c)>0:
            f0_mean,f0_std,f0_min,f0_max = np.mean(f0_c),np.std(f0_c),np.min(f0_c),np.max(f0_c)
        else: f0_mean=f0_std=f0_min=f0_max=0.0
    except: f0_mean=f0_std=f0_min=f0_max=0.0
    mfcc = np.mean(librosa.feature.mfcc(y=y,sr=sr,n_mfcc=5),axis=1)
    return np.array([spec_cent,spec_bw,spec_roll,zcr,flatness,rms,
                     f0_mean,f0_std,f0_min,f0_max,*mfcc])

# smoke test
_vid = next(iter(labeled_audio))
_f = extract_features(audio_files[_vid])
assert _f is not None and _f.shape==(15,)
print(f"Smoke test OK: {_vid} → {_f.shape}")

In [ ]:
# Cell 4: Load model
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(15,64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64,32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32,16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16,1), nn.Sigmoid())
    def forward(self,x): return self.net(x)

model = Net()
sd = torch.load(MODEL_PATH, map_location='cpu')
model.load_state_dict(sd, strict=False)
model.eval()
print(f"Model loaded ✓ ({MODEL_PATH})")

In [ ]:
# Cell 5: Get label (1 if any L segment)
def get_label(vid):
    try:
        df = pd.read_csv(label_files[vid]['path'])
    except: return None
    if 'label' not in df.columns: return None
    return int((df['label']=='L').any())

# resume checkpoint
CKPT_FILE = f"{OUT_DIR}/eval_checkpoint.json"
records = []
done = set()
if Path(CKPT_FILE).exists():
    records = json.loads(Path(CKPT_FILE).read_text())
    done = {r['vid'] for r in records}
    print(f"Resuming: {len(done)} done")

all_vids = sorted(labeled_audio)
todo = [v for v in all_vids if v not in done]
print(f"Evaluating {len(todo)} of {len(all_vids)} videos …")

for vid in tqdm(todo):
    lab = get_label(vid)
    if lab is None: continue
    feats = extract_features(audio_files[vid])
    if feats is None: continue
    with torch.no_grad():
        prob = model(torch.tensor(feats,dtype=torch.float32).unsqueeze(0)).item()
    records.append({'vid':vid,'prob':prob,'pred':int(prob>0.5),
                    'label':lab,'split':label_files[vid]['split']})
    if len(records)%25==0:
        Path(CKPT_FILE).write_text(json.dumps(records))

Path(CKPT_FILE).write_text(json.dumps(records))

all_true = [r['label'] for r in records]
all_pred = [r['pred'] for r in records]
print(f"\n=== OVERALL ({len(records)} videos) ===")
print(f"F1: {f1_score(all_true,all_pred):.4f}  P: {precision_score(all_true,all_pred):.4f}  R: {recall_score(all_true,all_pred):.4f}")
for sp in ('val','train','test','all','unknown'):
    t=[r['label'] for r in records if r['split']==sp]
    p=[r['pred']  for r in records if r['split']==sp]
    if t: print(f"  {sp}: F1={f1_score(t,p):.4f}  n={len(t)}")

In [ ]:
# Cell 6: Save results
results = {
    'f1': float(f1_score(all_true,all_pred)),
    'precision': float(precision_score(all_true,all_pred)),
    'recall': float(recall_score(all_true,all_pred)),
    'n_videos': len(records),
    'pos_rate': float(np.mean(all_true)),
    'pred_pos_rate': float(np.mean(all_pred)),
}
rp = f"{OUT_DIR}/eval_results.json"
cp = f"{OUT_DIR}/per_video_predictions.csv"
with open(rp,'w') as f: json.dump(results,f,indent=2)
pd.DataFrame(records).to_csv(cp,index=False)
print(f"Saved: {rp}")
print(f"Saved: {cp}")
print(json.dumps(results,indent=2))